# Extract keyword using NER

## Import library

In [59]:
!pip install spacy
!python -m spacy download en_core_web_sm
!pip install pandas pyarrow keybert sentence-transformers scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 26.8 MB/s eta 0:00:0000:010:01

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [60]:
import spacy
import pandas as pd
import os
from keybert import KeyBERT

# Set to project root
os.chdir('/home/martin/technical-news')

/home/martin/technical-news/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [61]:
# Load spaCy NER model
ner_model = spacy.load("en_core_web_sm")

# Initialize KeyBERT model with a lightweight model
kw_model = KeyBERT(model="all-MiniLM-L6-v2")

In [62]:
# Load parquet file - latest data from Jan 21
linkedin_data = pd.read_parquet("data/raw/linkedin/linkedin_scraping_data_20260120_203615.parquet")
print(f"Total records: {len(linkedin_data)}")
print(linkedin_data.head())

Total records: 127
                                            post_url          activity_id  \
0  https://www.linkedin.com/feed/update/urn:li:ac...  7358971409227792384   
1  https://www.linkedin.com/feed/update/urn:li:ac...  7356343466827661314   
2  https://www.linkedin.com/feed/update/urn:li:ac...  7389740123669721088   
3  https://www.linkedin.com/feed/update/urn:li:ac...  7417950714561384451   
4  https://www.linkedin.com/feed/update/urn:li:ac...  7418595295606767616   

                                                text            topic  \
0  What people think will improve AI applications...      [technical]   
1  13 years. 6 books. All metrics are flawed, but...  [non-technical]   
2  Had a great time chatting with Lenny Rachitsky...      [technical]   
3  An article in Le Monde about my startup AMI La...  [non-technical]   
4  We release Action100M, the hero behind VL-JEPA...      [technical]   

  supported_industry  keywords  reactions  comments  reposts  \
0       [Techno

In [74]:
import re
from collections import Counter

results = []

# Optional: very simple stop words if you don't have nltk/spacy stopwords loaded
simple_stop = {'the','a','an','and','or','but','is','are','was','were','this','that','these','those','in','on','at','to','of','for','with','by','from','as','about','like','through','over','between','into','during','including','toward','against','after','before','up','down','out','off','around','among','below','above','under','beside','behind','next','near','far','here','there','now','then','today','tomorrow','yesterday','good','new','first','last','long','great','little','own','other','old','right','big','high','different','small','large','next','early','young','important','few','public','bad','same','able'}

for i, record in linkedin_data.iterrows():
    text = record['text']
    
    # Quick clean - helps almost all methods
    text_clean = re.sub(r'https?://\S+|@\w+|#[\w_]+|\s+', ' ', text.lower()).strip()
    text_clean = re.sub(r'[^\w\s]', '', text_clean)  # remove punctuation
    
    # ────────────────────────────────────────────────
    # Option A: Improved NER - take several relevant entities
    # ────────────────────────────────────────────────
    doc = ner_model(text)
    entities = []
    seen_lower = set()
    for ent in doc.ents:
        if ent.label_ in ['PERSON','ORG','PRODUCT','GPE','NORP','FAC','EVENT','WORK_OF_ART']:
            ent_lower = ent.text.strip().lower()
            # Only add if not seen before (case-insensitive deduplication)
            if ent_lower not in seen_lower:
                entities.append(ent.text.strip())
                seen_lower.add(ent_lower)
    
    ner_keywords = entities[:4] if entities else []
    ner_type = doc.ents[0].label_ if doc.ents else None   # keep first type for reference

    # ────────────────────────────────────────────────
    # Option B: Much better KeyBERT call
    # ────────────────────────────────────────────────
    keybert_keywords = []
    keybert_str = None
    keybert_top_score = None
    try:
        extracted = kw_model.extract_keywords(
            text,                       # better to use original text (not cleaned)
            keyphrase_ngram_range=(1, 3),
            stop_words='english',
            use_mmr=True,
            diversity=0.4,              # 0.3–0.6 usually good
            top_n=5                     # get 5 → pick 2–4 later
        )
        
        if extracted:
            # Filter very short/generic words if you want
            filtered = [kw for kw, score in extracted if len(kw.split()) >= 1 and score > 0.30]
            keybert_keywords = [kw for kw, _ in extracted[:4]]
            keybert_top_score = extracted[0][1] if extracted else None
            
    except Exception as e:
        print(f"KeyBERT error on {i}: {str(e)}")
        keybert_keywords = []

    # ────────────────────────────────────────────────
    # REMOVE DUPLICATES: Filter out KeyBERT keywords that already appear in NER
    # ────────────────────────────────────────────────
    ner_lower = {kw.lower() for kw in ner_keywords}
    keybert_unique = [kw for kw in keybert_keywords if kw.lower() not in ner_lower]
    
    # Create display strings
    ner_keywords_str = ", ".join(ner_keywords) if ner_keywords else None
    keybert_str = ", ".join(keybert_unique) if keybert_unique else None

    # ────────────────────────────────────────────────
    # Optional C: Very simple statistical fallback (if both above weak)
    # ────────────────────────────────────────────────
    words = [w for w in text_clean.split() if w not in simple_stop and len(w) > 2]
    if len(words) > 5:
        cnt = Counter(words)
        common = [w for w, c in cnt.most_common(6) if c >= 1]
        bigrams = [' '.join(words[j:j+2]) for j in range(len(words)-1) if words[j] in common[:4] or words[j+1] in common[:4]]
        fallback = ", ".join(set(common[:3] + bigrams[:2]))  # mix unigrams + bigrams
    else:
        fallback = None

    results.append({
        'record_id': i,
        'post_text_short': text[:80] + '...' if len(text) > 80 else text,
        'ner_keywords': ner_keywords_str,
        'ner_type_first': ner_type,
        'keybert_keywords': keybert_str,
        'keybert_top_score': round(keybert_top_score,4) if keybert_top_score else None,
        'fallback_stat': fallback
    })

# ──── create and show dataframe ────
results_df = pd.DataFrame(results)

print("\n=== KEYWORD EXTRACTION SUMMARY (All Duplicates Removed) ===")
print(f"Total records: {len(results_df)}")
print(f"With NER keywords: {results_df['ner_keywords'].notna().sum()}")
print(f"With KeyBERT keywords (after removing duplicates): {results_df['keybert_keywords'].notna().sum()}")
print(f"With fallback keywords: {results_df['fallback_stat'].notna().sum()}")

print("\n=== COMPARISON TABLE ===")
print(results_df[['record_id','ner_keywords','keybert_keywords','fallback_stat']].head(15).to_string(index=False))

print("\n=== ANALYSIS ===")
both = ((results_df['ner_keywords'].notna()) & (results_df['keybert_keywords'].notna())).sum()
only_ner = ((results_df['ner_keywords'].notna()) & (results_df['keybert_keywords'].isna())).sum()
only_keybert = ((results_df['ner_keywords'].isna()) & (results_df['keybert_keywords'].notna())).sum()
neither = ((results_df['ner_keywords'].isna()) & (results_df['keybert_keywords'].isna())).sum()

print(f"Both methods found (non-duplicate) keywords: {both} records")
print(f"Only NER found keywords: {only_ner} records")
print(f"Only KeyBERT found keywords: {only_keybert} records")
print(f"Neither method found keywords: {neither} records")


=== KEYWORD EXTRACTION SUMMARY (All Duplicates Removed) ===
Total records: 127
With NER keywords: 116
With KeyBERT keywords (after removing duplicates): 127
With fallback keywords: 122

=== COMPARISON TABLE ===
 record_id                                                    ner_keywords                                                                                               keybert_keywords                                                                    fallback_stat
         0                                                         AI, Q&A         retrieval performance improvement, ai applications vs, application performance, vector database debate                                 what, applications what, they, data, what people
         1                                                            None                                        13 years books, years books metrics, books metrics flawed, love writing                                        books all, years books, years